# Feature Extraction in Text — Full Pipeline

This notebook runs the complete, hand-built pipeline end to end on real
documents scraped from Hacker News (`hackernews_dataset.csv`, produced by
`scrape_hackernews.py`), falling back to the small curated `demo.csv` corpus
if that file isn't present:

1. Load raw text from CSV
2. Preprocess (lowercase, strip punctuation/numbers, tokenize, remove
   stopwords, stem)
3. POS tagging
4. Syntactic features (subject/verb/object, noun/verb counts)
5. Semantic features (rule-based lexical categories)
6. Morphological analysis
7. Lemmatization (vs. stemming)
8. Vocabulary + Bag of Words / Binary BoW
9. One-hot encoding (tokens and POS tags)
10. TF-IDF + document cosine similarity
11. N-grams

All logic comes from `preprocessing.py`, `linguistic_features.py`,
`syntactic_features.py`, `semantic_features.py`, and `features.py` in this
project — no scikit-learn, nltk, or spaCy.

## 1. Setup

In [2]:
import pandas as pd

from preprocessing import (
    preprocess,
    lowercase,
    remove_punctuation_and_numbers,
    tokenize,
    remove_stopwords,
    stem_word,
    stem_tokens,
)
from linguistic_features import (
    pos_tag,
    pos_tag_word,
    morphological_analysis,
    lemmatize_word,
    lemmatize_tokens,
)
from syntactic_features import syntactic_analysis, syntactic_feature_matrix
from semantic_features import semantic_feature_matrix, cosine_similarity_matrix
from features import (
    build_vocabulary,
    bag_of_words,
    binary_bow,
    tf_idf,
    n_grams,
    one_hot_encode_tokens,
    one_hot_encode_categories,
)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 2. Load raw text from CSV

Reads `hackernews_dataset.csv` (real scraped Hacker News stories, one per
row in a `text` column) if it exists, otherwise falls back to the small
curated `demo.csv` corpus.

In [3]:
from pathlib import Path

csv_path = "hackernews_dataset.csv" if Path("hackernews_dataset.csv").exists() else "demo.csv"
data = pd.read_csv(csv_path)
corpus = data["text"].tolist()

print(f"Loaded {len(corpus)} documents from {csv_path}\n")
for i, doc in enumerate(corpus):
    print(f"Doc {i}: {doc}")

Loaded 50 documents from hackernews_dataset.csv

Doc 0: The Therapeutic Potential of Apigenin (2018)
Doc 1: Interactive Physics
Doc 2: The Economics of the Intelligence Frontier
Doc 3: Packslip – signed release manifest for archives, installers, or executables
Doc 4: The Innovative HP Computer Behind a Hedge Fund Pioneer
Doc 5: Federal judge again rules against Musk's xAI
Doc 6: TIL: Using Blender with coding agents on macOS
Doc 7: Show HN: MarkFlowy – A open source Markdown editor rebuilt for large documents. I maintain MarkFlowy, a desktop Markdown editor for Windows, macOS, and Linux. I’ve spent the past two months rebuilding its editor core, mainly to improve performance with larger documents. The changes are part of v0.100.0. In testing, a 2 MB Markdown document opened in about a second. I’d be interested to see how it performs with other people’s documents and hardware. Opening speed was one part of the work. I also worked on editing long documents and copying and cutting large s

## 3. Preprocessing

`preprocess()` runs lowercase -> strip punctuation/numbers -> tokenize ->
remove stopwords -> stem, in one call. We also keep a lighter "raw tokens"
version per document (lowercased and tokenized, but *not* stopword-stripped
or stemmed) for the linguistic analysis steps below, since POS tagging and
morphology need function words and full word forms to work with.


In [4]:
processed_docs = [preprocess(doc) for doc in corpus]
raw_tokens_per_doc = [
    tokenize(remove_punctuation_and_numbers(lowercase(doc))) for doc in corpus
]

print("=== Tokens after full preprocessing (stopwords removed, stemmed) ===")
for i, tokens in enumerate(processed_docs):
    print(f"Doc {i}: {tokens}")


=== Tokens after full preprocessing (stopwords removed, stemmed) ===
Doc 0: ['therapeutic', 'potential', 'apigenin']
Doc 1: ['interactive', 'physic']
Doc 2: ['economic', 'intelligence', 'frontier']
Doc 3: ['packslip', 'sign', 'release', 'manifest', 'archiv', 'installer', 'executabl']
Doc 4: ['innovative', 'hp', 'computer', 'behind', 'hedge', 'fund', 'pioneer']
Doc 5: ['federal', 'judge', 'rul', 'musk', 's', 'xai']
Doc 6: ['til', 'using', 'blender', 'cod', 'agent', 'maco']
Doc 7: ['show', 'hn', 'markflowy', 'open', 'source', 'markdown', 'editor', 'rebuilt', 'large', 'document', 'maintain', 'markflowy', 'desktop', 'markdown', 'editor', 'window', 'maco', 'linux', 've', 'spent', 'past', 'two', 'month', 'rebuild', 'editor', 'core', 'main', 'improve', 'performance', 'larger', 'document', 'chang', 'part', 'v', 'test', 'mb', 'markdown', 'document', 'open', 'second', 'd', 'interest', 'see', 'how', 'perform', 'other', 'people', 's', 'document', 'hardware', 'open', 'spe', 'one', 'part', 'work', '

## 4. POS Tagging

Rule-based tagging (closed-class lexicon + suffix rules) on the raw tokens
of every document, shown as one combined table.


In [5]:
pos_rows = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    for word, tag in pos_tag(tokens):
        pos_rows.append({"doc": doc_index, "word": word, "pos_tag": tag})

pos_df = pd.DataFrame(pos_rows)
pos_df


,doc,word,pos_tag
0,0,the,DET
1,0,therapeutic,ADJ
2,0,potential,ADJ
3,0,of,PREP
4,0,apigenin,NOUN
...,...,...,...
1548,48,os,NOUN
1549,49,ssh,NOUN
1550,49,on,PREP
1551,49,the,DET


## 5. Syntactic Features

Heuristic subject/verb/object extraction (first nominal before/after the
first verb) plus noun/verb counts, one row per document.

In [6]:
syntactic_frames = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    tagged = pos_tag(tokens)
    doc_syntax = syntactic_analysis(tagged)
    doc_syntax.insert(0, "doc", doc_index)
    syntactic_frames.append(doc_syntax)

syntactic_df = pd.concat(syntactic_frames, ignore_index=True)
syntactic_df

,doc,subject,verb,object,token_count,noun_count,verb_count,has_subject_verb_object
0,0,None,None,None,5,1,0,False
1,1,None,None,None,2,1,0,False
2,2,None,None,None,6,3,0,False
3,3,packslip,signed,release,9,6,1,True
4,4,None,None,None,9,6,0,False
5,5,None,None,None,8,6,0,False
6,6,til,using,blender,8,4,2,True
7,7,show,rebuilding,its,196,105,25,True
8,8,how,upcoming,german,9,7,1,True
9,9,None,None,None,9,9,0,False


## 6. Semantic Features

Counts of tokens per hand-written lexical category (action, animal, place,
object, positive_description, descriptive) from `SEMANTIC_LEXICON`.

In [7]:
semantic_df = semantic_feature_matrix(raw_tokens_per_doc)
semantic_df

,action,animal,place,object,positive_description,descriptive
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0
5,0,0,0,0,0,0
6,0,0,0,0,0,0
7,0,0,0,0,0,0
8,0,0,0,0,0,0
9,0,0,0,0,0,0


## 7. Morphological Analysis

Per-word shape and inflection features (length, vowel/consonant counts,
prefix/suffix, plural/gerund/past-tense/comparative/superlative flags) for
every document, combined into one table.

In [8]:
morph_frames = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    doc_morph = morphological_analysis(tokens)
    doc_morph.insert(0, "doc", doc_index)
    morph_frames.append(doc_morph)

morph_df = pd.concat(morph_frames, ignore_index=True)
morph_df


,doc,word,length,num_vowels,num_consonants,prefix3,suffix3,is_capitalized,is_plural,is_gerund,is_past_tense,is_comparative,is_superlative
0,0,the,3,1,2,the,the,False,False,False,False,False,False
1,0,therapeutic,11,5,6,the,tic,False,False,False,False,False,False
2,0,potential,9,4,5,pot,ial,False,False,False,False,False,False
3,0,of,2,1,1,of,of,False,False,False,False,False,False
4,0,apigenin,8,4,4,api,nin,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1548,48,os,2,1,1,os,os,False,False,False,False,False,False
1549,49,ssh,3,0,3,ssh,ssh,False,False,False,False,False,False
1550,49,on,2,1,1,on,on,False,False,False,False,False,False
1551,49,the,3,1,2,the,the,False,False,False,False,False,False


## 8. Lemmatization vs. Stemming

Both reduce a word to a base form, but the stemmer just chops suffixes
(sometimes producing fragments that aren't real words), while the
lemmatizer aims to return an actual dictionary word. Comparison shown on
stopword-free tokens from every document.

In [9]:
lemma_rows = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    filtered = remove_stopwords(tokens)
    for word in filtered:
        lemma_rows.append({
            "doc": doc_index,
            "word": word,
            "stem": stem_word(word),
            "lemma": lemmatize_word(word),
        })

lemma_df = pd.DataFrame(lemma_rows)
lemma_df


,doc,word,stem,lemma
0,0,therapeutic,therapeutic,therapeutic
1,0,potential,potential,potential
2,0,apigenin,apigenin,apigenin
3,1,interactive,interactive,interactive
4,1,physics,physic,physic
...,...,...,...,...
1038,48,style,style,style
1039,48,web,web,web
1040,48,os,os,os
1041,49,ssh,ssh,ssh


## 9. Vocabulary

Built from the fully preprocessed (stopword-free, stemmed) tokens.

In [10]:
vocab = build_vocabulary(processed_docs)
print(f"Vocabulary size: {len(vocab)}")
vocab


Vocabulary size: 626


['able',
 'acces',
 'account',
 'accurate',
 'actual',
 'add',
 'addict',
 'advertisement',
 'affect',
 'afraid',
 'age',
 'agent',
 'ai',
 'all',
 'allow',
 'almost',
 'already',
 'also',
 'alway',
 'american',
 'annotate',
 'anonymou',
 'answer',
 'anti',
 'any',
 'anycast',
 'anymore',
 'anyth',
 'apigenin',
 'app',
 'appeal',
 'appreciate',
 'apprentice',
 'archiv',
 'around',
 'arthriti',
 'artificial',
 'ask',
 'association',
 'attach',
 'autism',
 'automatical',
 'available',
 'award',
 'awkward',
 'back',
 'bad',
 'ban',
 'bandwhich',
 'bann',
 'bas',
 'basical',
 'because',
 'becom',
 'become',
 'behavior',
 'behind',
 'berner',
 'beyond',
 'big',
 'binary',
 'blender',
 'block',
 'book',
 'borrow',
 'break',
 'broader',
 'bug',
 'built',
 'byt',
 'calculator',
 'call',
 'came',
 'can',
 'cannot',
 'card',
 'cas',
 'caus',
 'ccp',
 'cdn',
 'censorship',
 'certain',
 'cgroup',
 'chang',
 'chat',
 'chatgpt',
 'china',
 'chinese',
 'claude',
 'cloud',
 'cod',
 'code',
 'codebase'

## 10. Bag of Words

In [11]:
bow_df = bag_of_words(processed_docs, vocab)
bow_df


,able,acces,account,accurate,actual,add,addict,advertisement,affect,afraid,age,agent,ai,all,allow,...,word,work,world,worri,would,wouldn,wrapp,written,wrong,xai,xi,year,yes,young,zero
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,6,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 11. Binary Bag of Words

In [12]:
binary_df = binary_bow(processed_docs, vocab)
binary_df


,able,acces,account,accurate,actual,add,addict,advertisement,affect,afraid,age,agent,ai,all,allow,...,word,work,world,worri,would,wouldn,wrapp,written,wrong,xai,xi,year,yes,young,zero
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 12. One-Hot Encoding

Two flavors: one-hot per token *position* in a document (preserves order,
unlike bag-of-words), and a generic one-hot encoding of category labels
(here, the POS tags of Doc 0).

In [13]:
print("=== One-hot encoding of tokens (Doc 0) ===")
one_hot_tokens_df = one_hot_encode_tokens(processed_docs[0], vocab)
one_hot_tokens_df


=== One-hot encoding of tokens (Doc 0) ===


,able,acces,account,accurate,actual,add,addict,advertisement,affect,afraid,age,agent,ai,all,allow,...,word,work,world,worri,would,wouldn,wrapp,written,wrong,xai,xi,year,yes,young,zero
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [14]:
print("=== One-hot encoding of POS tags (Doc 0) ===")
pos_tags_doc0 = [tag for _, tag in pos_tag(raw_tokens_per_doc[0])]
one_hot_pos_df = one_hot_encode_categories(pos_tags_doc0)
one_hot_pos_df


=== One-hot encoding of POS tags (Doc 0) ===


,ADJ,DET,NOUN,PREP
0,0,1,0,0
1,1,0,0,0
2,1,0,0,0
3,0,0,0,1
4,0,0,1,0


## 13. TF-IDF

In [15]:
tfidf_df = tf_idf(processed_docs, vocab)
tfidf_df.round(3)


,able,acces,account,accurate,actual,add,addict,advertisement,affect,afraid,age,agent,ai,all,allow,...,word,work,world,worri,would,wouldn,wrapp,written,wrong,xai,xi,year,yes,young,zero
0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
4,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.912,0.000,0.000,0.000,0.000,0.000
6,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.219,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
7,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,23.472,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
8,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,3.912,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
9,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.303,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


### Document Similarity (TF-IDF Cosine)

Pairwise cosine similarity between documents' TF-IDF vectors — lexical
overlap, not deep semantic meaning.

In [16]:
cosine_similarity_matrix(tfidf_df).round(3)

,doc_0,doc_1,doc_2,doc_3,doc_4,doc_5,doc_6,doc_7,doc_8,doc_9,doc_10,doc_11,doc_12,doc_13,doc_14,...,doc_35,doc_36,doc_37,doc_38,doc_39,doc_40,doc_41,doc_42,doc_43,doc_44,doc_45,doc_46,doc_47,doc_48,doc_49
doc_0,1.000,0.0,0.00,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,...,0.000,0.000,0.000,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.00,0.000,0.0
doc_1,0.000,1.0,0.00,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,...,0.000,0.000,0.000,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.00,0.000,0.0
doc_2,0.000,0.0,1.00,0.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,...,0.000,0.000,0.000,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.16,0.000,0.0
doc_3,0.000,0.0,0.00,1.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,...,0.000,0.000,0.000,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.00,0.000,0.0
doc_4,0.000,0.0,0.00,0.0,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,...,0.000,0.000,0.000,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.00,0.000,0.0
doc_5,0.000,0.0,0.00,0.0,0.000,1.000,0.000,0.012,0.000,0.000,0.000,0.0,0.000,0.000,0.000,...,0.000,0.000,0.000,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.00,0.000,0.0
doc_6,0.000,0.0,0.00,0.0,0.000,0.000,1.000,0.021,0.000,0.000,0.000,0.0,0.000,0.000,0.000,...,0.000,0.000,0.268,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.00,0.000,0.0
doc_7,0.000,0.0,0.00,0.0,0.000,0.012,0.021,1.000,0.011,0.012,0.000,0.0,0.009,0.000,0.000,...,0.017,0.000,0.000,0.0,0.000,0.000,0.0,0.021,0.010,0.000,0.0,0.000,0.00,0.034,0.0
doc_8,0.000,0.0,0.00,0.0,0.000,0.000,0.000,0.011,1.000,0.000,0.000,0.0,0.000,0.000,0.000,...,0.000,0.000,0.000,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.000,0.00,0.000,0.0
doc_9,0.000,0.0,0.00,0.0,0.000,0.000,0.000,0.012,0.000,1.000,0.000,0.0,0.000,0.000,0.000,...,0.000,0.000,0.050,0.0,0.000,0.105,0.0,0.000,0.000,0.069,0.0,0.000,0.00,0.075,0.0


## 14. N-grams

Bigrams and trigrams for every document, built from the fully preprocessed
tokens.

In [17]:
for i, tokens in enumerate(processed_docs):
    print(f"Doc {i}")
    print("  Bigrams: ", n_grams(tokens, 2))
    print("  Trigrams:", n_grams(tokens, 3))


Doc 0
  Bigrams:  [('therapeutic', 'potential'), ('potential', 'apigenin')]
  Trigrams: [('therapeutic', 'potential', 'apigenin')]
Doc 1
  Bigrams:  [('interactive', 'physic')]
  Trigrams: []
Doc 2
  Bigrams:  [('economic', 'intelligence'), ('intelligence', 'frontier')]
  Trigrams: [('economic', 'intelligence', 'frontier')]
Doc 3
  Bigrams:  [('packslip', 'sign'), ('sign', 'release'), ('release', 'manifest'), ('manifest', 'archiv'), ('archiv', 'installer'), ('installer', 'executabl')]
  Trigrams: [('packslip', 'sign', 'release'), ('sign', 'release', 'manifest'), ('release', 'manifest', 'archiv'), ('manifest', 'archiv', 'installer'), ('archiv', 'installer', 'executabl')]
Doc 4
  Bigrams:  [('innovative', 'hp'), ('hp', 'computer'), ('computer', 'behind'), ('behind', 'hedge'), ('hedge', 'fund'), ('fund', 'pioneer')]
  Trigrams: [('innovative', 'hp', 'computer'), ('hp', 'computer', 'behind'), ('computer', 'behind', 'hedge'), ('behind', 'hedge', 'fund'), ('hedge', 'fund', 'pioneer')]
Doc 5


## Summary

```
raw text -> lowercase -> remove punctuation/numbers -> tokenize
         -> remove stopwords -> stem/lemmatize
         -> [POS tagging / syntactic / semantic / morphological on the side]
         -> feature extraction (BoW / one-hot / TF-IDF / cosine similarity / n-grams)
```

Every step above is implemented from scratch in `preprocessing.py`,
`linguistic_features.py`, `syntactic_features.py`, `semantic_features.py`,
and `features.py` using only `numpy` and `pandas`. See `report.md` for the
theory behind each step.